<a href="https://colab.research.google.com/github/LIKHITHA-NAGRAJ/Water_Quality_Prediction_ML_project/blob/main/Water_safe_unsafe_prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
"""
AI Water Quality Prediction — Starter Script
==============================================
Works with the Kaggle "Water Potability" dataset (water_potability.csv)
Columns typically: ph, Hardness, Solids, Chloramines, Sulfate, Conductivity,
                    Organic_carbon, Trihalomethanes, Turbidity, Potability

If your downloaded CSV has different column names, just update DATA_PATH
and the TARGET_COL variable below.
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, roc_auc_score, roc_curve
)

# ---------------------------------------------------------
# 1. LOAD DATA
# ---------------------------------------------------------
DATA_PATH = "/content/drive/MyDrive/archive (2).zip"   # <-- update path of dataset
TARGET_COL = "Potability"            # <-- update if your target column has a different name

df = pd.read_csv(DATA_PATH)

print("Shape:", df.shape)
print("\nFirst 5 rows:")
print(df.head())

print("\nColumn info:")
print(df.info())

# ---------------------------------------------------------
# 2. EDA — MISSING VALUES
# ---------------------------------------------------------
print("\nMissing values per column:")
print(df.isnull().sum())

missing_pct = df.isnull().mean() * 100
print("\nMissing value %:")
print(missing_pct[missing_pct > 0])

# ---------------------------------------------------------
# 3. EDA — CLASS BALANCE (safe vs unsafe)
# ---------------------------------------------------------
print("\nClass balance:")
print(df[TARGET_COL].value_counts())
print(df[TARGET_COL].value_counts(normalize=True) * 100)

plt.figure(figsize=(5, 4))
sns.countplot(x=TARGET_COL, data=df)
plt.title("Class Balance: Safe (1) vs Unsafe (0)")
plt.savefig("class_balance.png", bbox_inches="tight")
plt.close()
print("Saved class_balance.png")

# ---------------------------------------------------------
# 4. EDA — CORRELATION HEATMAP
# ---------------------------------------------------------
plt.figure(figsize=(10, 8))
sns.heatmap(df.corr(), annot=True, fmt=".2f", cmap="coolwarm")
plt.title("Correlation Heatmap")
plt.savefig("correlation_heatmap.png", bbox_inches="tight")
plt.close()
print("Saved correlation_heatmap.png")

# ---------------------------------------------------------
# 5. HANDLE MISSING VALUES
# ---------------------------------------------------------
# Water Potability dataset commonly has missing values in ph, Sulfate,
# and Trihalomethanes. Median imputation is a safe default for skewed
# numeric data like this.
imputer = SimpleImputer(strategy="median")
X = df.drop(columns=[TARGET_COL])
y = df[TARGET_COL]

X_imputed = pd.DataFrame(imputer.fit_transform(X), columns=X.columns)

# ---------------------------------------------------------
# 6. TRAIN/TEST SPLIT
# ---------------------------------------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X_imputed, y, test_size=0.2, random_state=42, stratify=y
)

print(f"\nTrain size: {X_train.shape[0]}, Test size: {X_test.shape[0]}")

# ---------------------------------------------------------
# 7. SCALE FEATURES
# ---------------------------------------------------------
# Logistic Regression and SVM are sensitive to feature scale;
# Random Forest doesn't need it but scaling doesn't hurt it either.
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# ---------------------------------------------------------
# 8. TRAIN MODELS
# ---------------------------------------------------------
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, class_weight="balanced"),
    "Random Forest": RandomForestClassifier(n_estimators=200, random_state=42, class_weight="balanced"),
    "SVM": SVC(probability=True, class_weight="balanced"),
}

results = []

for name, model in models.items():
    model.fit(X_train_scaled, y_train)
    y_pred = model.predict(X_test_scaled)
    y_proba = model.predict_proba(X_test_scaled)[:, 1]

    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_proba)

    results.append({
        "Model": name, "Accuracy": acc, "Precision": prec,
        "Recall": rec, "F1-Score": f1, "ROC-AUC": auc
    })

    print(f"\n===== {name} =====")
    print(classification_report(y_test, y_pred))
    print("Confusion Matrix:")
    print(confusion_matrix(y_test, y_pred))

# ---------------------------------------------------------
# 9. COMPARE MODELS
# ---------------------------------------------------------
results_df = pd.DataFrame(results).sort_values(by="F1-Score", ascending=False)
print("\n===== Model Comparison =====")
print(results_df.to_string(index=False))

best_model_name = results_df.iloc[0]["Model"]
print(f"\nBest model by F1-Score: {best_model_name}")

# ---------------------------------------------------------
# 10. ROC CURVES (all models on one plot)
# ---------------------------------------------------------
plt.figure(figsize=(7, 6))
for name, model in models.items():
    y_proba = model.predict_proba(X_test_scaled)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, y_proba)
    auc = roc_auc_score(y_test, y_proba)
    plt.plot(fpr, tpr, label=f"{name} (AUC={auc:.3f})")

plt.plot([0, 1], [0, 1], "k--", label="Random")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curves — Model Comparison")
plt.legend()
plt.savefig("roc_curves.png", bbox_inches="tight")
plt.close()
print("Saved roc_curves.png")

print("\nDone. Check the saved PNGs and the printed metrics above.")

Shape: (3276, 10)

First 5 rows:
         ph    Hardness        Solids  Chloramines     Sulfate  Conductivity  \
0       NaN  204.890455  20791.318981     7.300212  368.516441    564.308654   
1  3.716080  129.422921  18630.057858     6.635246         NaN    592.885359   
2  8.099124  224.236259  19909.541732     9.275884         NaN    418.606213   
3  8.316766  214.373394  22018.417441     8.059332  356.886136    363.266516   
4  9.092223  181.101509  17978.986339     6.546600  310.135738    398.410813   

   Organic_carbon  Trihalomethanes  Turbidity  Potability  
0       10.379783        86.990970   2.963135           0  
1       15.180013        56.329076   4.500656           0  
2       16.868637        66.420093   3.055934           0  
3       18.436524       100.341674   4.628771           0  
4       11.558279        31.997993   4.075075           0  

Column info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3276 entries, 0 to 3275
Data columns (total 10 columns):
 #   

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
"""
AI Water Quality Prediction — IMPROVED Training Script
========================================================
Improvements over the baseline script:
  1. KNN imputation instead of median (uses similar rows to estimate missing values)
  2. SMOTE oversampling on the training set (actually balances classes, not just reweights)
  3. Hyperparameter tuning via GridSearchCV for Random Forest
  4. Adds Gradient Boosting as a fourth model
  5. Saves the best model + scaler + imputer to disk so predict_manual.py can use them

Honest expectation: this dataset has near-zero linear correlation between
every feature and Potability (see your correlation heatmap). These changes
should give a real but modest improvement — don't expect 90%+ accuracy;
that would actually be a red flag on this data, not a win.
"""

import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.impute import KNNImputer
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, roc_auc_score
)
from imblearn.over_sampling import SMOTE

DATA_PATH = "/content/drive/MyDrive/archive (2).zip"
TARGET_COL = "Potability"

# ---------------------------------------------------------
# 1. LOAD + IMPUTE (KNN-based, smarter than median)
# ---------------------------------------------------------
df = pd.read_csv(DATA_PATH)
X = df.drop(columns=[TARGET_COL])
y = df[TARGET_COL]
feature_names = list(X.columns)

print("Imputing missing values with KNNImputer (this can take a few seconds)...")
imputer = KNNImputer(n_neighbors=5)
X_imputed = pd.DataFrame(imputer.fit_transform(X), columns=feature_names)

# ---------------------------------------------------------
# 2. TRAIN/TEST SPLIT (before any resampling — test set stays untouched/real)
# ---------------------------------------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X_imputed, y, test_size=0.2, random_state=42, stratify=y
)

# ---------------------------------------------------------
# 3. SCALE
# ---------------------------------------------------------
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# ---------------------------------------------------------
# 4. SMOTE — oversample minority class on TRAINING data only
#    (never apply SMOTE to test data — that would leak synthetic
#    samples into your evaluation and inflate results artificially)
# ---------------------------------------------------------
print("Applying SMOTE to balance the training set...")
print("Before SMOTE:", dict(pd.Series(y_train).value_counts()))
smote = SMOTE(random_state=42)
X_train_bal, y_train_bal = smote.fit_resample(X_train_scaled, y_train)
print("After SMOTE:", dict(pd.Series(y_train_bal).value_counts()))

# ---------------------------------------------------------
# 5. HYPERPARAMETER TUNING — Random Forest
# ---------------------------------------------------------
print("\nTuning Random Forest hyperparameters (GridSearchCV, this takes a bit)...")
rf_param_grid = {
    "n_estimators": [200, 400],
    "max_depth": [None, 10, 20],
    "min_samples_split": [2, 5],
    "min_samples_leaf": [1, 2],
}
rf_grid = GridSearchCV(
    RandomForestClassifier(random_state=42),
    rf_param_grid, cv=3, scoring="f1", n_jobs=-1
)
rf_grid.fit(X_train_bal, y_train_bal)
print("Best RF params:", rf_grid.best_params_)

# ---------------------------------------------------------
# 6. TRAIN ALL MODELS ON THE BALANCED TRAINING SET
# ---------------------------------------------------------
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "SVM": SVC(probability=True),
    "Random Forest (tuned)": rf_grid.best_estimator_,
    "Gradient Boosting": GradientBoostingClassifier(random_state=42),
}

results = []
trained_models = {}

for name, model in models.items():
    if name != "Random Forest (tuned)":  # already fitted above
        model.fit(X_train_bal, y_train_bal)
    trained_models[name] = model

    y_pred = model.predict(X_test_scaled)
    y_proba = model.predict_proba(X_test_scaled)[:, 1]

    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_proba)

    results.append({"Model": name, "Accuracy": acc, "Precision": prec, "Recall": rec, "F1-Score": f1, "ROC-AUC": auc})

    print(f"\n===== {name} =====")
    print(classification_report(y_test, y_pred))
    print("Confusion Matrix:")
    print(confusion_matrix(y_test, y_pred))

# ---------------------------------------------------------
# 7. COMPARE + PICK BEST
# ---------------------------------------------------------
results_df = pd.DataFrame(results).sort_values(by="F1-Score", ascending=False)
print("\n===== Model Comparison (Improved Pipeline) =====")
print(results_df.to_string(index=False))

best_model_name = results_df.iloc[0]["Model"]
best_model = trained_models[best_model_name]
print(f"\nBest model by F1-Score: {best_model_name}")

# ---------------------------------------------------------
# 8. SAVE MODEL + SCALER + IMPUTER FOR DEPLOYMENT
# ---------------------------------------------------------
joblib.dump(best_model, "best_water_model.pkl")
joblib.dump(scaler, "water_scaler.pkl")
joblib.dump(imputer, "water_imputer.pkl")
joblib.dump(feature_names, "water_feature_names.pkl")

print(f"\nSaved '{best_model_name}' to best_water_model.pkl")
print("Saved scaler to water_scaler.pkl")
print("Saved imputer to water_imputer.pkl")
print("Saved feature name order to water_feature_names.pkl")
print("\nYou can now run predict_manual.py to test your own water samples.")

Imputing missing values with KNNImputer (this can take a few seconds)...
Applying SMOTE to balance the training set...
Before SMOTE: {0: np.int64(1598), 1: np.int64(1022)}
After SMOTE: {0: np.int64(1598), 1: np.int64(1598)}

Tuning Random Forest hyperparameters (GridSearchCV, this takes a bit)...
Best RF params: {'max_depth': None, 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 400}

===== Logistic Regression =====
              precision    recall  f1-score   support

           0       0.62      0.50      0.56       400
           1       0.40      0.52      0.45       256

    accuracy                           0.51       656
   macro avg       0.51      0.51      0.50       656
weighted avg       0.53      0.51      0.52       656

Confusion Matrix:
[[201 199]
 [123 133]]

===== SVM =====
              precision    recall  f1-score   support

           0       0.69      0.67      0.68       400
           1       0.50      0.52      0.51       256

    accuracy    

In [4]:
"""
Predict Water Potability for a Manually-Entered Sample
=========================================================
Run this AFTER running water_quality_improved.py at least once
(it needs the saved .pkl files that script produces).

Usage:
    python predict_manual.py

You'll be prompted for each water quality parameter. Press Enter
without typing a value if you don't know it — the same KNN imputer
used during training will estimate it for you.
"""

import joblib
import numpy as np
import pandas as pd

MODEL_PATH = "best_water_model.pkl"
SCALER_PATH = "water_scaler.pkl"
IMPUTER_PATH = "water_imputer.pkl"
FEATURES_PATH = "water_feature_names.pkl"

# Typical real-world ranges, shown as a guide while entering values
FEATURE_HINTS = {
    "ph": "0-14, WHO safe range ~6.5-8.5",
    "Hardness": "mg/L, typically 50-350",
    "Solids": "ppm (total dissolved solids), typically 300-50000",
    "Chloramines": "ppm, typically 0-13",
    "Sulfate": "mg/L, typically 100-500",
    "Conductivity": "\u00b5S/cm, typically 180-750",
    "Organic_carbon": "ppm, typically 2-30",
    "Trihalomethanes": "\u00b5g/L, typically 0-125",
    "Turbidity": "NTU, typically 1-7",
}


def get_float_input(feature_name):
    hint = FEATURE_HINTS.get(feature_name, "")
    prompt = f"  {feature_name} ({hint}) [Enter to skip]: "
    raw = input(prompt).strip()
    if raw == "":
        return np.nan
    try:
        return float(raw)
    except ValueError:
        print("    Not a number, treating as unknown.")
        return np.nan


def main():
    print("Loading saved model, scaler, and imputer...")
    model = joblib.load(MODEL_PATH)
    scaler = joblib.load(SCALER_PATH)
    imputer = joblib.load(IMPUTER_PATH)
    feature_names = joblib.load(FEATURES_PATH)

    print("\n=== Enter your water sample values ===")
    print("(Leave blank and press Enter for any value you don't know)\n")

    values = {}
    for feature in feature_names:
        values[feature] = get_float_input(feature)

    sample_df = pd.DataFrame([values], columns=feature_names)

    # Same preprocessing pipeline as training: impute -> scale
    sample_imputed = pd.DataFrame(imputer.transform(sample_df), columns=feature_names)
    sample_scaled = scaler.transform(sample_imputed)

    prediction = model.predict(sample_scaled)[0]
    probability = model.predict_proba(sample_scaled)[0]

    print("\n" + "=" * 45)
    if prediction == 1:
        print(f"PREDICTION: POTABLE (safe to drink)")
    else:
        print(f"PREDICTION: NOT POTABLE (unsafe)")
    print(f"Confidence -> Safe: {probability[1]*100:.1f}%  |  Unsafe: {probability[0]*100:.1f}%")
    print("=" * 45)
    print("\nNote: this is a student-project model trained on a dataset with")
    print("very weak feature-target correlation — treat this as a demo of")
    print("the ML pipeline, not as real water safety guidance.")


if __name__ == "__main__":
    main()

Loading saved model, scaler, and imputer...

=== Enter your water sample values ===
(Leave blank and press Enter for any value you don't know)

  ph (0-14, WHO safe range ~6.5-8.5) [Enter to skip]: 7
  Hardness (mg/L, typically 50-350) [Enter to skip]: 12
  Solids (ppm (total dissolved solids), typically 300-50000) [Enter to skip]: 8000
  Chloramines (ppm, typically 0-13) [Enter to skip]: 5
  Sulfate (mg/L, typically 100-500) [Enter to skip]: 300
  Conductivity (µS/cm, typically 180-750) [Enter to skip]: 300
  Organic_carbon (ppm, typically 2-30) [Enter to skip]: 5
  Trihalomethanes (µg/L, typically 0-125) [Enter to skip]: 28
  Turbidity (NTU, typically 1-7) [Enter to skip]: 6

PREDICTION: NOT POTABLE (unsafe)
Confidence -> Safe: 17.6%  |  Unsafe: 82.4%

Note: this is a student-project model trained on a dataset with
very weak feature-target correlation — treat this as a demo of
the ML pipeline, not as real water safety guidance.
